## Analysis
This notebook contains the code to generate the main tables and plots for the paper.
It expects the results to be collected in `.csv` files as created by [`results_processing.ipynb`](./results_processing.ipynb).

In [ ]:
import os
import json
import yaml
from pathlib import Path

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from termcolor import colored, cprint

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path("notebooks")
FIG_DIR = NOTEBOOK_DIR / "assets"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv("../data/processed_results_all.csv")
model_info = pd.read_csv("../data/models_info.csv")

In [ ]:
def clean_model_name(model_name: str) -> str:
	if "gpt" in model_name or "/o3" in model_name:
		return model_name.split("/")[-1].strip()
	else:
		return model_name.split("openai/")[-1].split("/")[-1].strip().replace("thinking", "").replace("Thinking", "").replace("preview", "").strip(" -")
	
def get_model_family(model_name: str) -> tuple:
	"""
	Returns model family name + version number
	"""
	if "gpt-oss" in model_name:
		return "GPT-OSS", None
	elif "Kimi-" in model_name:
		return "Kimi", "K2.X"
	elif "GLM" in model_name:
		return "GLM", None
	elif "gemini" in model_name:
		return "Gemini", "3.X" if "3" in model_name else ("2.5" if "2.5" in model_name else model_name.split("gemini-")[-1].split("-")[0])
	elif "gpt" in model_name:
		return "GPT", model_name.split("gpt-")[-1].split("-")[0]
	elif "gemma" in model_name:
		return "Gemma", "4"
	elif "Qwen" in model_name:
		return "Qwen", model_name.split("Qwen")[-1].split("-")[0]
	elif "deepseek" in model_name.lower():
		return "DeepSeek", "V4"
	else:
		return model_name, None

MODEL_FAMILY_COLORS = {
    "Qwen": "#9759CD",      # purple
    "Gemma": "#2DADF2",     # blue
    "Gemini": "#3F76ED",    # blue
    "Kimi": "#D75252",      # red
    "GPT": "#00AC64",       # OpenAI green
    "GPT-OSS": "#9DC98B",   # OpenAI green
    "OpenAI": "#004D40",    # OpenAI green
    "o3": "#004D40",        # OpenAI green
    "DeepSeek": "#3729DA",  # orange
    "GLM": "#E9881A",       # remaining family
}

MODEL_FAMILY_ORDER = list(MODEL_FAMILY_COLORS)

def get_sorted_families(families):
    return sorted(
        families,
        key=lambda fam: (
            MODEL_FAMILY_ORDER.index(fam)
            if fam in MODEL_FAMILY_ORDER
            else len(MODEL_FAMILY_ORDER),
            fam,
        ),
    )

MODEL_LABEL_MODE = "family_version"  # Options: "model_name", "family_version"
MODEL_LABEL_POINT_FALLBACK = "best_accuracy"  # Options: "best_accuracy", "random", "none"
MODEL_LABEL_RANDOM_SEED = 85
MODEL_LABEL_POINT_CONFIG = {
    "accuracy_vs_cost": {
        "text": {
            "DeepSeek V4": "DeepSeek-V4-Pro",
            "GLM None": "GLM-4.7",
            "GPT 5": "gpt-5",
            "GPT 5.2": None,
            "GPT 5.4": "gpt-5.4",
            "GPT 5.5": "gpt-5.5",
            "GPT-oss None": "gpt-oss-120b",
            "Gemini 2.5": "gemini-2.5-flash",
            "Gemini 3.X": "gemini-3-flash",
            "Gemma 4": "gemma-4-E4B-it",
            "Kimi K2.X": "Kimi-K2.6",
            "Qwen 3": "Qwen3-VL-30B-A3B",
            "Qwen 3.5": "Qwen3.5-35B-A3B",
        },
        "multi": {
            "GPT 5": "gpt-5",
            "GPT 5.2": "gpt-5.2",
            "GPT 5.4": "gpt-5.4",
			"GPT 5.5": "gpt-5.5",
            "Gemini 2.5": "gemini-2.5-flash",
            "Gemini 3.X": "gemini-3.1-pro",
            "Gemma 4": "gemma-4-26B-A4B-it",
            "Kimi K2.X": "Kimi-K2.5",
            "Qwen 3": "Qwen3-VL-235B-A22B",
            "Qwen 3.5": "Qwen3.5-122B-A10B",
        },
        "image_gen": {
            "GPT image": "gpt-image-1.5",
            "Gemini 2.5": "gemini-2.5-flash-image",
            "Gemini 3.X": "gemini-3.1-flash-image",
        },
    },
    "accuracy_vs_aa_index": {
        "text": {
            "DeepSeek V4": "DeepSeek-V4-Pro",
            "GLM None": "GLM-4.7",
            "GPT 5": "gpt-5-mini",
            "GPT 5.2": None,
            "GPT 5.4": "gpt-5.4",
			"GPT 5.5": "gpt-5.5",
            "GPT-OSS None": "gpt-oss-20b",
            "Gemini 2.5": "gemini-2.5-pro",
            "Gemini 3.X": "gemini-3-flash",
            "Gemma 4": "gemma-4-E4B-it",
            "Kimi K2.X": "Kimi-K2.6",
            "Qwen 3": "Qwen3-VL-30B-A3B",
            "Qwen 3.5": "Qwen3.5-122B-A10B",
        },
        "multi": {
            "GPT 5": "gpt-5",
            "GPT 5.2": "gpt-5.2",
            "GPT 5.4": "gpt-5.4",
			"GPT 5.5": "gpt-5.5",
            "Gemini 2.5": "gemini-2.5-pro",
            "Gemini 3.X": "gemini-3-flash",
            "Gemma 4": "gemma-4-26B-A4B-it",
            "Kimi K2.X": "Kimi-K2.5",
            "Qwen 3": "Qwen3-VL-30B-A3B",
            "Qwen 3.5": "Qwen3.5-397B-A17B",
        },
    },
}

def get_family_colors(families):
    fallback_colors = sns.color_palette("Set2", n_colors=max(len(families), 1)).as_hex()
    return {
        fam: MODEL_FAMILY_COLORS.get(fam, fallback_colors[i % len(fallback_colors)])
        for i, fam in enumerate(families)
    }


def format_model_label(row):
    if MODEL_LABEL_MODE == "model_name":
        return row["model_name"]
    if MODEL_LABEL_MODE == "family_version":
        family = row["model_family"]
        version = row["model_version"]
        if pd.isna(version) or str(version).lower() == "none":
            return family
        return f"{family} {version}"
    raise ValueError(f"Unknown MODEL_LABEL_MODE: {MODEL_LABEL_MODE}")

def select_label_point(g, plot_key, qtype, series_name, rng=None):
    missing = object()
    selector = (
        MODEL_LABEL_POINT_CONFIG
        .get(plot_key, {})
        .get(qtype, {})
        .get(series_name, missing)
    )

    if selector is None or str(selector).lower() == "none":
        return None

    if selector is not missing:
        match = g[g["model_name"] == selector]
        if match.empty:
            options = ", ".join(g["model_name"].astype(str).tolist())
            raise ValueError(
                f"No model_name={selector!r} for {plot_key}/{qtype}/{series_name}. "
                f"Available values: {options}"
            )
        return match.iloc[0]

    if MODEL_LABEL_POINT_FALLBACK == "none":
        return None
    if MODEL_LABEL_POINT_FALLBACK == "random":
        if rng is None:
            rng = np.random.default_rng(MODEL_LABEL_RANDOM_SEED)
        return g.iloc[rng.integers(len(g))]
    if MODEL_LABEL_POINT_FALLBACK == "best_accuracy":
        return g.loc[g["accuracy_pct"].idxmax()]
    raise ValueError(f"Unknown MODEL_LABEL_POINT_FALLBACK: {MODEL_LABEL_POINT_FALLBACK}")

def slugify(value):
    return "".join(
        ch if ch.isalnum() else "-"
        for ch in str(value).lower()
    ).strip("-")

PLOT_FIGSIZE_WITH_LEGEND = (8, 5)
PLOT_FIGSIZE_NO_LEGEND = (7, 5)
PLOT_AXES_RECT_INCHES = (0.85, 0.72, 5.35, 3.45)  # left, bottom, width, height

def set_fixed_plot_surface(fig, ax):
    fig_width, fig_height = fig.get_size_inches()
    left, bottom, width, height = PLOT_AXES_RECT_INCHES
    ax.set_position([
        left / fig_width,
        bottom / fig_height,
        width / fig_width,
        height / fig_height,
    ])

def add_prefix(model_name: str) -> str:
	if "GLM" in model_name:
		return "zai-org/" + model_name
	elif "gemma" in model_name:
		return "google/" + model_name
	elif "Qwen" in model_name:
		return "Qwen/" + model_name
	elif "deepseek" in model_name.lower():
		return "deepseek-ai/" + model_name
	elif "kimi" in model_name.lower():
		return "moonshotai/" + model_name

	elif "gpt-oss" in model_name:
		return "openai/" + model_name
	else:
		print(f"Unknown model name: {model_name}")
		return model_name

In [ ]:
# assert not any((df.groupby(["model_name", "qid"])["score"].count() < 4))

pass_at_4 = (
	df.groupby(["model_name", "qtype", "qid"])["score"]
	.any()
	.groupby(["model_name", "qtype"])
	.mean()
	.rename("pass@4")
)

df = df.groupby(["model_name", "qtype"])[["score", "error", "price", "out_tokens"]].agg(
	{
		"score": ["mean", "count"],
		"error": ["sum"],
		"price": ["mean", "sum"],
		"out_tokens": ["mean"]
	}
)
df[("score", "pass@4")] = pass_at_4
df["valid_ratio"] = (df["score"]["count"] - df["error"]["sum"]) / df["score"]["count"]

In [ ]:
# Flatten multi-index columns
df.columns = ["_".join(col).strip() for col in df.columns.values]
df = df.reset_index()
df.rename(columns={"score_mean": "accuracy", "score_pass@4": "pass@4", "price_mean": "avg_price_per_sample"}, inplace=True)

df = pd.merge(df, model_info[["model_name", "size"]], on="model_name", how="left")
df["model_name"] = df["model_name"].apply(clean_model_name)
df["model_family"], df["model_version"] = zip(*df["model_name"].apply(get_model_family))

# Set pass@4 to None for image generation
df.loc[df["qtype"] == "image_gen", "pass@4"] = None

df.sort_values(["qtype", "accuracy"], inplace=True)

## Table

In [ ]:
QTYPE_TABLE_CONFIG = {
    'text': {
        'label': 'Text-only',
        'metrics': [
            ('accuracy', 'mean@4', True, 'pct'),
            ('pass@4', 'pass@4', True, 'pct'),
            ('out_tokens_mean', 'out-tks', False, 'int'),
            ('price_per_100_samples', '\\$/100', False, 'price'),
        ],
    },
    'multi': {
        'label': 'Multi-to-text',
        'metrics': [
            ('accuracy', 'mean@4', True, 'pct'),
            ('pass@4', 'pass@4', True, 'pct'),
            ('out_tokens_mean', 'out-tks', False, 'int'),
            ('price_per_100_samples', '\\$/100', False, 'price'),
        ],
    },
    'image_gen': {
        'label': 'Image-gen',
        'metrics': [
            ('accuracy', 'mean@1', True, 'pct'),
            ('price_per_100_samples', '\\$/100', False, 'price'),
        ],
    },
}

def latex_escape(value):
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}',
    }
    return ''.join(replacements.get(ch, ch) for ch in str(value))

def format_table_value(value, fmt):
    if pd.isna(value):
        return '-'
    def trim_zeros(text):
        return text.rstrip('0').rstrip('.')
    if fmt == 'pct':
        return trim_zeros(f'{100 * value:.2f}')
    if fmt == 'int':
        return f'{value:.0f}'
    if fmt == 'price':
        return rf'\${trim_zeros(f"{value:.2f}")}'
    return trim_zeros(f'{value:.2f}')

def colored_table_cell(value, column_values, higher_is_better, fmt):
    if pd.isna(value):
        return '-'
    clean_values = [v for v in column_values if not pd.isna(v)]
    text = format_table_value(value, fmt)
    if len(clean_values) <= 1 or min(clean_values) == max(clean_values):
        return text
    normalized = (value - min(clean_values)) / (max(clean_values) - min(clean_values))
    score = normalized if higher_is_better else 1 - normalized
    green_pct = int(round(15 + 70 * score))
    return rf'\cellcolor{{green!{green_pct}!red!25!white}}{{{text}}}'

def build_results_latex_table(results_df):
    table_df = results_df.copy()
    table_df['price_per_100_samples'] = table_df['avg_price_per_sample'] * 100 
    table_index = table_df.set_index(['model_name', 'qtype'])

    text_accuracy = (
        table_df[table_df['qtype'] == 'text']
        .set_index('model_name')['accuracy']
    )
    models = sorted(
        table_df['model_name'].dropna().unique(),
        key=lambda model: text_accuracy.get(model, -np.inf),
        reverse=True,
    )

    column_values = {}
    for qtype, config in QTYPE_TABLE_CONFIG.items():
        for metric, _, _, _ in config['metrics']:
            vals = []
            for model in models:
                if (model, qtype) in table_index.index:
                    vals.append(table_index.loc[(model, qtype), metric])
            column_values[(qtype, metric)] = vals

    column_spec = 'l' + 'r' * sum(
        len(config['metrics']) for config in QTYPE_TABLE_CONFIG.values()
    )
    macro_header = ['']
    cmidrules = []
    metric_header = ['Model']
    start_col = 2
    for config in QTYPE_TABLE_CONFIG.values():
        span = len(config['metrics'])
        macro_header.append(rf'\multicolumn{{{span}}}{{c}}{{\textbf{{{config["label"]}}}}}')
        cmidrules.append(rf'\cmidrule(lr){{{start_col}-{start_col + span - 1}}}')
        metric_header.extend(label for _, label, _, _ in config['metrics'])
        start_col += span

    row_end = r'\\'
    lines = [
        r'\begin{table}[t]',
        r'\centering',
        r'\small',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\begin{{tabular}}{{{column_spec}}}',
        r'\toprule',
        ' & '.join(macro_header) + ' ' + row_end,
        ' '.join(cmidrules),
        ' & '.join(metric_header) + ' ' + row_end,
        r'\midrule',
    ]

    for model in models:
        row = [latex_escape(model)]
        for qtype, config in QTYPE_TABLE_CONFIG.items():
            has_row = (model, qtype) in table_index.index
            for metric, _, higher_is_better, fmt in config['metrics']:
                value = table_index.loc[(model, qtype), metric] if has_row else np.nan
                row.append(
                    colored_table_cell(
                        value,
                        column_values[(qtype, metric)],
                        higher_is_better,
                        fmt,
                    )
                )
        lines.append(' & '.join(row) + ' ' + row_end)

    lines.extend([
        r'\bottomrule',
        r'\end{tabular}',
        r'\caption{Model results by evaluation subset. Accuracy and pass rates are shown as percentages. Lower token and price values are better.}',
        r'\label{tab:leaderboard}',
        r'\end{table}',
    ])
    return '\n'.join(lines)

latex_table = build_results_latex_table(df)
(FIG_DIR / 'model_results_table.tex').write_text(latex_table)
print("Saved LaTeX table to:", FIG_DIR / 'model_results_table.tex')

---

## Plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ------------------------------------------------------------------
# CLEANING / PREP
# ------------------------------------------------------------------

for qtype in df["qtype"].unique():
    plot_df = df[df["qtype"] == qtype].copy()

    plot_df["accuracy_pct"] = plot_df["accuracy"] * 100
    plot_df["avg_price_per_100_samples"] = plot_df["avg_price_per_sample"] * 100

    # Group key = manufacturer/family + version
    plot_df["series"] = (
        plot_df["model_family"].astype(str)
        + " "
        + plot_df["model_version"].astype(str)
    )

    plot_df = plot_df.dropna(
        subset=["accuracy_pct", "avg_price_per_100_samples"]
    ).copy()

    # Sort lines by model size if available, otherwise by cost
    plot_df["sort_key"] = np.where(
        plot_df["size"].notna(),
        plot_df["size"],
        plot_df["avg_price_per_100_samples"]
    )

    # ------------------------------------------------------------------
    # STYLING
    # ------------------------------------------------------------------
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WITH_LEGEND)

    families = get_sorted_families(plot_df["model_family"].dropna().unique())
    family_colors = get_family_colors(families)

    # ------------------------------------------------------------------
    # DRAW CONNECTED SERIES
    # ------------------------------------------------------------------
    for series_name, g in plot_df.groupby("series"):
        g = g.sort_values("sort_key")

        fam = g["model_family"].iloc[0]
        color = family_colors.get(fam, "gray")

        # connected line
        ax.plot(
            g["avg_price_per_100_samples"],
            g["accuracy_pct"],
            lw=1.8,
            alpha=0.8,
            color=color,
            zorder=1
        )

        # scatter points
        ax.scatter(
            g["avg_price_per_100_samples"],
            g["accuracy_pct"],
            s=60,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=0.95,
            zorder=2
        )

    # ------------------------------------------------------------------
    # LABEL ONE CONFIGURED POINT OF EACH SERIES
    # ------------------------------------------------------------------
    label_rng = np.random.default_rng(MODEL_LABEL_RANDOM_SEED)
    for series_name, g in plot_df.groupby("series"):
        label_point = select_label_point(
            g,
            plot_key="accuracy_vs_cost",
            qtype=qtype,
            series_name=series_name,
            rng=label_rng,
        )
        if label_point is None:
            continue
        color = family_colors.get(label_point["model_family"], "gray")

        ax.annotate(
            format_model_label(label_point),
            xy=(label_point["avg_price_per_100_samples"], label_point["accuracy_pct"]),
            xytext=(8, 0),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
            ha="left",
            va="center",
            alpha=0.9,
            color=color,
            path_effects=[
                path_effects.withStroke(linewidth=0.65, foreground="white")
            ]
        )

    # ------------------------------------------------------------------
    # AXES
    # ------------------------------------------------------------------
    ax.set_xscale("log")

    ax.set_xlabel("Average Cost per 100 Samples ($)", fontsize=14)
    ax.set_ylabel("Accuracy (%)", fontsize=14)
    ax.set_title(
        f"{qtype.upper().replace('_', '-')} - Accuracy vs. Cost",
        fontsize=18,
        weight="bold",
        pad=20
    )

    # ax.set_ylim(0, 100)

    # Format x-axis as dollars
    ax.xaxis.set_major_formatter(
        FuncFormatter(lambda x, pos: f"${x:.4f}" if x < 0.01 else f"${x:.2f}")
    )

    # ------------------------------------------------------------------
    # LEGEND (one entry per family)
    # ------------------------------------------------------------------
    handles = []
    labels = []

    for fam in families:
        h = plt.Line2D(
            [0], [0],
            color=family_colors[fam],
            marker="o",
            lw=2,
            markersize=7
        )
        handles.append(h)
        labels.append(fam)

    legend = ax.legend(
        handles,
        labels,
        title="Model Family",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=True
    )

    # ------------------------------------------------------------------
    # GRID / LAYOUT
    # ------------------------------------------------------------------
    ax.grid(True, which="major", alpha=0.25)
    ax.grid(True, which="minor", alpha=0.08)

    set_fixed_plot_surface(fig, ax)
    fig.savefig(FIG_DIR / f"{slugify(qtype)}_accuracy_vs_cost.pdf", bbox_inches="tight")

    legend.remove()
    fig.set_size_inches(PLOT_FIGSIZE_NO_LEGEND)
    set_fixed_plot_surface(fig, ax)
    fig.savefig(FIG_DIR / f"{slugify(qtype)}_accuracy_vs_cost_no_legend.pdf", bbox_inches="tight")

    fig.set_size_inches(PLOT_FIGSIZE_WITH_LEGEND)
    ax.legend(
        handles,
        labels,
        title="Model Family",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=True
    )
    set_fixed_plot_surface(fig, ax)
    plt.show()

---

## Accuracy vs. AA Intelligence Index

In [ ]:
import requests
import pandas as pd

aai_index_path = "../data/artificial_analysis_intelligence_index.csv"

if not os.path.exists(aai_index_path):
    API_KEY = os.getenv("ARTIFICIAL_ANALYSIS_API_KEY")

    url = "https://artificialanalysis.ai/api/v2/data/llms/models"
    headers = {"x-api-key": API_KEY}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    models = r.json()["data"]

    rows = []
    for m in models:
        evals = m.get("evaluations", {})

        rows.append({
            "model_id": m.get("id"),
            "model_name": m.get("name"),
            "creator": m.get("model_creator", {}).get("name"),
            "aa_intelligence_index": evals.get("artificial_analysis_intelligence_index"),
            "aa_coding_index": evals.get("artificial_analysis_coding_index"),
            "aa_math_index": evals.get("artificial_analysis_math_index"),
        })

    aai_index = pd.DataFrame(rows)
    aai_index.to_csv(aai_index_path, index=False)
else:
    aai_index = pd.read_csv(aai_index_path)
    aai_index_dict = {}
    for _, row in aai_index.iterrows():
        aai_index_dict[row["model_name"]] = row["aa_intelligence_index"]

In [ ]:
mapping = {
    # OpenAI reasoning models, medium effort where available
    "gpt-5.4-mini": aai_index_dict.get("GPT-5.4 mini (medium)", None),
    # "gpt-5.4-nano": aai_index_dict.get("GPT-5.4 nano (medium)", None),
    "gpt-5.2": aai_index_dict.get("GPT-5.2 (medium)", None),
    "gpt-5": aai_index_dict.get("GPT-5 (medium)", None),
    "gpt-5-mini": aai_index_dict.get("GPT-5 mini (medium)", None),
	# missing values, use proxy
    "gpt-5.4": (aai_index_dict["GPT-5.4 (xhigh)"] + aai_index_dict["GPT-5.4 (Non-reasoning)"]) / 2, # proxy for medium effort
    "gpt-5.5": aai_index_dict.get("GPT-5.5 (medium)", None), 
    "gpt-oss-120b": (aai_index_dict["gpt-oss-120B (high)"] + aai_index_dict["gpt-oss-120B (low)"]) / 2, # proxy for medium effort,
    "gpt-oss-20b": (aai_index_dict["gpt-oss-20B (high)"] + aai_index_dict["gpt-oss-20B (low)"]) / 2, # proxy for medium effort,

    # Google / Gemini / Gemma
    "gemini-3.1-pro": aai_index_dict.get("Gemini 3.1 Pro Preview", None),
    "gemini-3-flash": aai_index_dict.get("Gemini 3 Flash Preview (Reasoning)", None),
    "gemini-3.1-flash-lite": aai_index_dict.get("Gemini 3.1 Flash-Lite Preview", None),
    "gemini-2.5-pro": aai_index_dict.get("Gemini 2.5 Pro", None),
    "gemini-2.5-flash": aai_index_dict.get("Gemini 2.5 Flash (Reasoning)", None),
    "gemma-4-31B-it": aai_index_dict.get("Gemma 4 31B (Reasoning)", None),
    "gemma-4-26B-A4B-it": aai_index_dict.get("Gemma 4 26B A4B (Reasoning)", None),
    "gemma-4-E4B-it": aai_index_dict.get("Gemma 4 E4B (Reasoning)", None),
    "gemma-4-E2B-it": aai_index_dict.get("Gemma 4 E2B (Reasoning)", None),

    # Qwen reasoning models
    "Qwen3.5-397B-A17B": aai_index_dict.get("Qwen3.5 397B A17B (Reasoning)", None),
    "Qwen3.5-122B-A10B": aai_index_dict.get("Qwen3.5 122B A10B (Reasoning)", None),
    "Qwen3.5-35B-A3B": aai_index_dict.get("Qwen3.5 35B A3B (Reasoning)", None),
    "Qwen3-VL-235B-A22B": aai_index_dict.get("Qwen3 VL 235B A22B (Reasoning)", None),
    "Qwen3-Next-80B-A3B": aai_index_dict.get("Qwen3 Next 80B A3B (Reasoning)", None),
    "Qwen3-VL-30B-A3B": aai_index_dict.get("Qwen3 VL 30B A3B (Reasoning)", None),

    # Kimi
    "Kimi-K2.6": aai_index_dict.get("Kimi K2.6", None),
    "Kimi-K2.5": aai_index_dict.get("Kimi K2.5 (Reasoning)", None),
	
    # GLM
    "GLM-5.1": aai_index_dict.get("GLM-5.1 (Reasoning)", None),
    "GLM-4.7": aai_index_dict.get("GLM-4.7 (Reasoning)", None),
    "GLM-5": aai_index_dict.get("GLM-5 (Reasoning)", None),

    # DeepSeek
    "DeepSeek-V4-Pro": aai_index_dict.get("DeepSeek V4 Pro (Reasoning, High Effort)", None),
    "DeepSeek-V4-Flash": aai_index_dict.get("DeepSeek V4 Flash (Reasoning, High Effort)", None),
}

df["aa_intelligence_index"] = df["model_name"].map(mapping)

In [ ]:
for qtype in df["qtype"].unique():
    plot_df = df[df["qtype"] == qtype].copy()

    plot_df["accuracy_pct"] = plot_df["accuracy"] * 100

    # Group key = manufacturer/family + version
    plot_df["series"] = (
        plot_df["model_family"].astype(str)
        + " "
        + plot_df["model_version"].astype(str)
    )

    plot_df = plot_df.dropna(
        subset=["accuracy_pct", "aa_intelligence_index"]
    ).copy()

    # Sort lines by model size if available, otherwise by intelligence index
    plot_df["sort_key"] = np.where(
        plot_df["size"].notna(),
        plot_df["size"],
        plot_df["aa_intelligence_index"]
    )

    # ------------------------------------------------------------------
    # STYLING
    # ------------------------------------------------------------------
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WITH_LEGEND)

    families = get_sorted_families(plot_df["model_family"].dropna().unique())
    family_colors = get_family_colors(families)

    # ------------------------------------------------------------------
    # DRAW CONNECTED SERIES
    # ------------------------------------------------------------------
    for series_name, g in plot_df.groupby("series"):
        g = g.sort_values("sort_key")

        fam = g["model_family"].iloc[0]
        color = family_colors.get(fam, "gray")

        # connected line
        ax.plot(
            g["aa_intelligence_index"],
            g["accuracy_pct"],
            lw=1.8,
            alpha=0.8,
            color=color,
            zorder=1
        )

        # scatter points
        ax.scatter(
            g["aa_intelligence_index"],
            g["accuracy_pct"],
            s=60,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=0.95,
            zorder=2
        )

    # ------------------------------------------------------------------
    # LABEL ONE CONFIGURED POINT OF EACH SERIES
    # ------------------------------------------------------------------
    label_rng = np.random.default_rng(MODEL_LABEL_RANDOM_SEED)
    for series_name, g in plot_df.groupby("series"):
        label_point = select_label_point(
            g,
            plot_key="accuracy_vs_aa_index",
            qtype=qtype,
            series_name=series_name,
            rng=label_rng,
        )
        if label_point is None:
            continue
        color = family_colors.get(label_point["model_family"], "gray")

        ax.annotate(
            format_model_label(label_point),
            xy=(label_point["aa_intelligence_index"], label_point["accuracy_pct"]),
            xytext=(8, 0),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
            ha="left",
            va="center",
            alpha=0.9,
            color=color,
            path_effects=[
                path_effects.withStroke(linewidth=0.65, foreground="white")
            ]
        )

    # ------------------------------------------------------------------
    # AXES
    # ------------------------------------------------------------------
    ax.set_xlabel("Artificial Analysis Intelligence Index", fontsize=14)
    ax.set_ylabel("Accuracy (%)", fontsize=14)
    ax.set_title(
        f"{qtype.upper()} - Accuracy vs. AA Intelligence Index",
        fontsize=18,
        weight="bold",
        pad=20
    )

    # ax.set_ylim(0, 100)

    # ------------------------------------------------------------------
    # LEGEND (one entry per family)
    # ------------------------------------------------------------------
    handles = []
    labels = []

    for fam in families:
        h = plt.Line2D(
            [0], [0],
            color=family_colors[fam],
            marker="o",
            lw=2,
            markersize=7
        )
        handles.append(h)
        labels.append(fam)

    legend = ax.legend(
        handles,
        labels,
        title="Model Family",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=True
    )

    # ------------------------------------------------------------------
    # GRID / LAYOUT
    # ------------------------------------------------------------------
    ax.grid(True, which="major", alpha=0.25)
    ax.grid(True, which="minor", alpha=0.08)

    set_fixed_plot_surface(fig, ax)
    fig.savefig(FIG_DIR / f"{slugify(qtype)}_accuracy_vs_aa_index.pdf", bbox_inches="tight")

    legend.remove()
    fig.set_size_inches(PLOT_FIGSIZE_NO_LEGEND)
    set_fixed_plot_surface(fig, ax)
    fig.savefig(FIG_DIR / f"{slugify(qtype)}_accuracy_vs_aa_index_no_legend.pdf", bbox_inches="tight")

    fig.set_size_inches(PLOT_FIGSIZE_WITH_LEGEND)
    ax.legend(
        handles,
        labels,
        title="Model Family",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=True
    )
    set_fixed_plot_surface(fig, ax)
    plt.show()

---
### Shared Failure Modes

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

QID_HEATMAP_QTYPES = {
    "text": "Text-only",
    "multi": "Multi-to-text",
    "image_gen": "Image-gen",
}
QID_SORT_BY_CLUSTER = False  # Set True to cluster qids by similar model-performance patterns.
QID_HEATMAP_ASCENDING = True  # True puts harder problems first when QID_SORT_BY_CLUSTER is False.
QID_CLUSTER_METRIC = "correlation"
QID_CLUSTER_METHOD = "average"
QID_HEATMAP_CELL_WIDTH = 0.13
QID_HEATMAP_CELL_HEIGHT = 0.15
QID_HEATMAP_MIN_FIGSIZE = (7, 4)
QID_HEATMAP_MAX_FIGSIZE = (22, 12)
QID_HEATMAP_WIDTH_PADDING = 3.0
QID_HEATMAP_HEIGHT_PADDING = 1.8

soft_red_green_cmap = LinearSegmentedColormap.from_list(
    "soft_red_green",
    ["#f3b5b5", "#fff8f0", "#b8e3b2"],
)


def get_qid_heatmap_order(qid_heatmap):
    model_order = qid_heatmap.mean(axis=1).sort_values(ascending=False).index
    qid_heatmap = qid_heatmap.loc[model_order]

    if QID_SORT_BY_CLUSTER:
        from scipy.cluster.hierarchy import leaves_list, linkage
        from scipy.spatial.distance import pdist

        qid_matrix = qid_heatmap.T.astype(float)
        qid_matrix = qid_matrix.fillna(qid_heatmap.mean(axis=1)).fillna(qid_heatmap.stack().mean())

        if len(qid_matrix) > 1:
            qid_distances = pdist(qid_matrix, metric=QID_CLUSTER_METRIC)
            if not np.isfinite(qid_distances).all():
                qid_distances = pdist(qid_matrix, metric="euclidean")
            qid_linkage = linkage(qid_distances, method=QID_CLUSTER_METHOD)
            qid_order = qid_matrix.index[leaves_list(qid_linkage)]
        else:
            qid_order = qid_matrix.index
    else:
        qid_sort_df = pd.DataFrame({
            "mean_accuracy": qid_heatmap.mean(axis=0),
        })
        qid_sort_df["qid_numeric"] = pd.to_numeric(qid_sort_df.index, errors="coerce")
        qid_sort_df["qid_string"] = qid_sort_df.index.astype(str)
        qid_sort_df = qid_sort_df.sort_values(
            ["mean_accuracy", "qid_numeric", "qid_string"],
            ascending=[QID_HEATMAP_ASCENDING, True, True],
        )
        qid_order = qid_sort_df.index

    return qid_heatmap.loc[:, qid_order]


def get_qid_heatmap_figsize(qid_heatmap):
    width = QID_HEATMAP_WIDTH_PADDING + QID_HEATMAP_CELL_WIDTH * qid_heatmap.shape[1]
    height = QID_HEATMAP_HEIGHT_PADDING + QID_HEATMAP_CELL_HEIGHT * qid_heatmap.shape[0]
    width = min(max(width, QID_HEATMAP_MIN_FIGSIZE[0]), QID_HEATMAP_MAX_FIGSIZE[0])
    height = min(max(height, QID_HEATMAP_MIN_FIGSIZE[1]), QID_HEATMAP_MAX_FIGSIZE[1])
    return width, height


qid_results_raw = pd.read_csv("../data/processed_results_all.csv")
qid_results_raw["model_name"] = qid_results_raw["model_name"].apply(clean_model_name)

for qtype, qtype_label in QID_HEATMAP_QTYPES.items():
    qtype_results = qid_results_raw[qid_results_raw["qtype"] == qtype].copy()
    if qtype_results.empty:
        continue

    qid_accuracy = (
        qtype_results.groupby(["model_name", "qid"], as_index=False)["score"]
        .mean()
        .rename(columns={"score": "accuracy"})
    )

    qid_heatmap = qid_accuracy.pivot(
        index="model_name",
        columns="qid",
        values="accuracy",
    )
    qid_heatmap = get_qid_heatmap_order(qid_heatmap)

    fig, ax = plt.subplots(figsize=get_qid_heatmap_figsize(qid_heatmap))
    sns.heatmap(
        qid_heatmap,
        ax=ax,
        cmap=soft_red_green_cmap,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Average accuracy"},
        linewidths=0,
        linecolor="white",
        mask=qid_heatmap.isna(),
        xticklabels=True,
        yticklabels=True,
    )

    ax.set_xlabel("Problem (qid)", fontsize=12)
    ax.set_ylabel("Model", fontsize=12)
    ax.set_title(
        f"{qtype_label} Per-Problem Accuracy by Model",
        fontsize=16,
        weight="bold",
        pad=12,
    )
    ax.tick_params(axis="x", labelrotation=90, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)

    plt.tight_layout()
    fig.savefig(FIG_DIR / f"{qtype}_qid_model_accuracy_heatmap.pdf", bbox_inches="tight")
    plt.show()

---

### Tool Use

In [ ]:
TOOL_USE_QTYPE_CONFIG = {
    "text": "Text-only",
}

TOOL_USE_METRICS = [
    ("accuracy", "acc", True, "pct"),
    ("out_tokens", "out-tks", False, "int"),
]

def load_tool_use_results(path, condition):
    data = pd.read_csv(path)
    data = data[data["qtype"].isin(TOOL_USE_QTYPE_CONFIG)].copy()
    data["model_name"] = data["model_name"].apply(clean_model_name)

    aggregated = (
        data.groupby(["model_name", "qtype"], as_index=False)
        .agg(
            accuracy=("score", "mean"),
            out_tokens=("out_tokens", "mean"),
            n_samples=("score", "count"),
        )
    )
    aggregated["condition"] = condition
    return aggregated

base_tool_use_df = load_tool_use_results("../data/processed_results_all.csv", "no_tools")
tools_tool_use_df = load_tool_use_results("../data/processed_results_tools.csv", "tools")

tool_use_comparison_df = base_tool_use_df.merge(
    tools_tool_use_df,
    on=["model_name", "qtype"],
    suffixes=("_no_tools", "_tools"),
)

tool_use_comparison_df["delta_accuracy_pp"] = 100 * (
    tool_use_comparison_df["accuracy_tools"]
    - tool_use_comparison_df["accuracy_no_tools"]
)
tool_use_comparison_df["delta_out_tokens"] = (
    tool_use_comparison_df["out_tokens_tools"]
    - tool_use_comparison_df["out_tokens_no_tools"]
)
tool_use_comparison_df = tool_use_comparison_df[
    [
        "qtype",
        "model_name",
        "accuracy_no_tools",
        "accuracy_tools",
        "delta_accuracy_pp",
        "out_tokens_no_tools",
        "out_tokens_tools",
        "delta_out_tokens",
        "n_samples_no_tools",
        "n_samples_tools",
    ]
].sort_values(["qtype", "delta_accuracy_pp", "model_name"], ascending=[True, False, True])

display(tool_use_comparison_df)

def trim_zeros(text):
    return text.rstrip("0").rstrip(".")

def format_tool_use_value(value, fmt):
    if pd.isna(value):
        return "-"
    if fmt == "pct":
        return trim_zeros(f"{100 * value:.2f}")
    if fmt == "int":
        return f"{value:.0f}"
    return trim_zeros(f"{value:.2f}")

def format_tool_use_delta(delta, fmt):
    if pd.isna(delta):
        return "-"
    sign = "+" if delta >= 0 else "-"
    abs_delta = abs(delta)
    if fmt == "pct":
        return f"{sign}{trim_zeros(f'{abs_delta:.2f}')}"
    if fmt == "int":
        return f"{sign}{abs_delta:.0f}"
    return f"{sign}{trim_zeros(f'{abs_delta:.2f}')}"

def tool_use_delta_cell(value, delta, higher_is_better, fmt):
    if pd.isna(value) or pd.isna(delta):
        return "-"
    improved = delta >= 0 if higher_is_better else delta <= 0
    color = "green" if improved else "red"
    base_text = format_tool_use_value(value, fmt)
    delta_text = format_tool_use_delta(delta, fmt)
    return rf"{base_text} {{\scriptsize\textcolor{{{color}!70!black}}{{({delta_text})}}}}"

def build_tool_use_latex_table(comparison_df):
    table_index = comparison_df.set_index(["model_name", "qtype"])
    text_accuracy = (
        comparison_df[comparison_df["qtype"] == "text"]
        .set_index("model_name")["accuracy_tools"]
    )
    models = sorted(
        comparison_df["model_name"].dropna().unique(),
        key=lambda model: text_accuracy.get(model, -np.inf),
        reverse=True,
    )

    column_spec = "l" + "r" * (len(TOOL_USE_QTYPE_CONFIG) * len(TOOL_USE_METRICS))
    macro_header = [""]
    cmidrules = []
    metric_header = ["Model"]
    start_col = 2
    for _, qtype_label in TOOL_USE_QTYPE_CONFIG.items():
        span = len(TOOL_USE_METRICS)
        macro_header.append(rf"\multicolumn{{{span}}}{{c}}{{\textbf{{{qtype_label}}}}}")
        cmidrules.append(rf"\cmidrule(lr){{{start_col}-{start_col + span - 1}}}")
        metric_header.extend(label for _, label, _, _ in TOOL_USE_METRICS)
        start_col += span

    row_end = r"\\"
    lines = [
        r"\centering",
        r"\footnotesize",
        r"\setlength{\tabcolsep}{4pt}",
        rf"\begin{{tabular}}{{{column_spec}}}",
        r"\toprule",
        " & ".join(macro_header) + " " + row_end,
        " ".join(cmidrules),
        " & ".join(metric_header) + " " + row_end,
        r"\midrule",
    ]

    for model in models:
        row = [latex_escape(model)]
        for qtype in TOOL_USE_QTYPE_CONFIG:
            has_row = (model, qtype) in table_index.index
            for metric, _, higher_is_better, fmt in TOOL_USE_METRICS:
                if not has_row:
                    row.append("-")
                    continue
                value = table_index.loc[(model, qtype), f"{metric}_tools"]
                if metric == "accuracy":
                    delta = table_index.loc[(model, qtype), "delta_accuracy_pp"]
                elif metric == "out_tokens":
                    delta = table_index.loc[(model, qtype), "delta_out_tokens"]
                else:
                    delta = np.nan
                row.append(tool_use_delta_cell(value, delta, higher_is_better, fmt))
        lines.append(" & ".join(row) + " " + row_end)

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{Tool-use comparison. Each cell shows the tool-use value, with the delta relative to the no-tool baseline in parentheses. Accuracy deltas are percentage points. Lower out-token deltas are better.}",
        r"\label{tab:tool-use-comparison}",
    ])
    return "\n".join(lines)

tool_use_latex_table = build_tool_use_latex_table(tool_use_comparison_df)
(FIG_DIR / "tool_use_comparison_table.tex").write_text(tool_use_latex_table)
print(f"Saved tool-use comparison LaTeX table to: {FIG_DIR / 'tool_use_comparison_table.tex'}")